### This notebook is intended to predict sql injection attacks
    Total data  : 14411
    Benign data : 6936+(1115)= 8051
    SQLI Data   : 86+77+474+6692+308+(290)=7927
    
#### At the end of notebook are results of testing

In [1]:

# import required packages

import glob
import time
import pandas as pd
# from xml.dom import minidom
from nltk import ngrams
from nltk.tokenize import sent_tokenize
import nltk
#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')
from nltk.stem import PorterStemmer
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize

In [2]:
import tensorflow.keras

In [3]:
import nltk
import pandas as pd
import os

### File_name: git_foospidy_payloads_fuzzing_code_db_sqli.txt
##### Comments :  
1. removed \n at the end of each line
2.  replaced %20 by space
#### Source: https://github.com/foospidy/payloads/tree/master/other/sqli

In [4]:
# preprocess sql data to have same format for all files

def clean_sqli_data(data):
    
    for i in range(len(data)):
        
        data[i]=data[i].replace('\n', '')
        data[i]=data[i].replace('%20', ' ')
        data[i]=data[i].replace('=', ' = ')
        data[i]=data[i].replace('((', ' (( ')
        data[i]=data[i].replace('))', ' )) ')
        data[i]=data[i].replace('(', ' ( ')
        data[i]=data[i].replace(')', ' ) ')
        data[i]=data[i].lower()
    return data

In [5]:
path='./data/sqlifuzzer.txt'

In [6]:
# read data from file

sql_lines_fuzzing=[]
f = open(path, "r")
for x in f:
    sql_lines_fuzzing.append(x)

In [7]:
sql_lines_fuzzing=clean_sqli_data(sql_lines_fuzzing) 

In [8]:
sql_lines_fuzzing[:15]

['2 and 456 = 678',
 '2 or 345 = 345',
 '2 order by 9999',
 '2 order by 1',
 '2/0 and 456 = 678',
 '2/1 or 345 = 345',
 '2/*f*/and/*f*/456 = 678',
 '2/*f*/or/*f*/345 = 345',
 "a' and '456' = '678",
 "a' or '345' = '345",
 "a' and 'fghi' = 'fghj'-- #",
 "a' or 'dfth' = 'dfth'-- #",
 "a' order by 9999-- #",
 "a' order by 1-- #",
 "a'and/*g*/456 = 678-- #"]

### File_Name: git_foospidy_payloads_other_sqli_camoufl4g3.txt
##### Comments : ok
#### Source: https://github.com/foospidy/payloads/tree/master/other/sqli

In [9]:
path='./data/camoufl4g3.txt'

In [10]:
# read data from file

sql_lines_camoufl4g3=[]
f = open(path, "r")
for x in f:
    sql_lines_camoufl4g3.append(x)

In [11]:
sql_lines_camoufl4g3[:15] # data before cleaning

["'-'\n",
 "' '\n",
 "'&'\n",
 "'^'\n",
 "'*'\n",
 "' or ''-'\n",
 "' or '' '\n",
 "' or ''&'\n",
 "' or ''^'\n",
 "' or ''*'\n",
 '"-"\n',
 '" "\n',
 '"&"\n',
 '"^"\n',
 '"*"\n']

In [12]:
sql_lines_camoufl4g3=clean_sqli_data(sql_lines_camoufl4g3)

In [13]:
sql_lines_camoufl4g3[:15]  # data after cleaning

["'-'",
 "' '",
 "'&'",
 "'^'",
 "'*'",
 "' or ''-'",
 "' or '' '",
 "' or ''&'",
 "' or ''^'",
 "' or ''*'",
 '"-"',
 '" "',
 '"&"',
 '"^"',
 '"*"']

### File_Name: git_foospidy_payloads_other_sqli_libinjection-bypasses.txt
##### Comments: Should Remove &()o1: from start of each sentence
#### Source: https://github.com/foospidy/payloads/tree/master/other/sqli

In [14]:
path='./data/libinjection-bypasses.txt'

In [15]:
# read data from file

sql_lines_bypasses=[]
f = open(path, "r")
for x in f:
    sql_lines_bypasses.append(x)

In [16]:
sql_lines_bypasses[:5] # before cleaning

['&()o1: select * from users where id=1 or (\\)=1 union select 1,@@VERSION -- 1\n',
 '&(.)o: select * from users where id=1 or (\\.)=1 union select 1,@@VERSION -- 1\n',
 '&(1&1: select * from users where id=1 or (\\+)=1 or 1=1 -- 1\n',
 '&(1)o: select * from users where id=1 or (1)=1 union select 1,banner from v$version where rownum=1 -- 1\n',
 '&(1UE: select * from users where id=1 or (\\+)=1 union select 1,@@VERSION -- 1\n']

In [17]:
sql_lines_bypasses=clean_sqli_data(sql_lines_bypasses)

In [18]:
sql_lines_bypasses[:5]  # data after cleaning

['& (  ) o1: select * from users where id = 1 or  ( \\ )  = 1 union select 1,@@version -- 1',
 '& ( . ) o: select * from users where id = 1 or  ( \\. )  = 1 union select 1,@@version -- 1',
 '& ( 1&1: select * from users where id = 1 or  ( \\+ )  = 1 or 1 = 1 -- 1',
 '& ( 1 ) o: select * from users where id = 1 or  ( 1 )  = 1 union select 1,banner from v$version where rownum = 1 -- 1',
 '& ( 1ue: select * from users where id = 1 or  ( \\+ )  = 1 union select 1,@@version -- 1']

In [19]:
# if don't want &(*)* sign in beginning of each sentence then run next code else don't

for i in range(len(sql_lines_bypasses)):
    sentence=sql_lines_bypasses[i]
    try:sql_lines_bypasses[i]=sentence.split(':')[1]
    except:pass
    

In [20]:
sql_lines_bypasses[:5]

[' select * from users where id = 1 or  ( \\ )  = 1 union select 1,@@version -- 1',
 ' select * from users where id = 1 or  ( \\. )  = 1 union select 1,@@version -- 1',
 ' select * from users where id = 1 or  ( \\+ )  = 1 or 1 = 1 -- 1',
 ' select * from users where id = 1 or  ( 1 )  = 1 union select 1,banner from v$version where rownum = 1 -- 1',
 ' select * from users where id = 1 or  ( \\+ )  = 1 union select 1,@@version -- 1']

### File_Name: git_foospidy_payloads_owasp_sqli.txt
##### Comments: ok
#### Source: https://github.com/foospidy/payloads/tree/master/other/sqli

In [21]:
path='./data/sqli_OSWAP.txt'

In [22]:
# read data from file

sql_lines_owasp=[]
f = open(path, "r")
for x in f:
    sql_lines_owasp.append(x)

In [23]:
sql_lines_owasp[0:15] # before cleaning

['\' or "\n',
 '-- or # \n',
 "' OR '1\n",
 "' OR 1 -- -\n",
 '" OR "" = "\n',
 '" OR 1 = 1 -- -\n',
 "' OR '' = '\n",
 "'='\n",
 "'LIKE'\n",
 "'=0--+\n",
 ' OR 1=1\n',
 "' OR 'x'='x\n",
 "' AND id IS NULL; --\n",
 "'''''''''''''UNION SELECT '2\n",
 'AND 1\n']

In [24]:
sql_lines_owasp=clean_sqli_data(sql_lines_owasp)

In [25]:
sql_lines_owasp[:15] # after cleaning

['\' or "',
 '-- or # ',
 "' or '1",
 "' or 1 -- -",
 '" or ""  =  "',
 '" or 1  =  1 -- -',
 "' or ''  =  '",
 "' = '",
 "'like'",
 "' = 0--+",
 ' or 1 = 1',
 "' or 'x' = 'x",
 "' and id is null; --",
 "'''''''''''''union select '2",
 'and 1']

### File_Name: git_seclist_fuzzing_sqli_Generic-SQLi.txt
##### comments: should we remove %20 by spaces because %20 equals to space
#### Source: https://github.com/danielmiessler/SecLists/tree/master/Fuzzing/SQLi

In [26]:
path='./data/Generic-SQLi.txt'

In [27]:
# read data from file

sql_lines_Generic=[]
f = open(path, "r")
for x in f:
    sql_lines_Generic.append(x)

In [28]:
sql_lines_Generic[:15] # before cleaning

[")%20or%20('x'='x\n",
 '%20or%201=1\n',
 "; execute immediate 'sel' || 'ect us' || 'er'\n",
 'benchmark(10000000,MD5(1))#\n',
 'update\n',
 '";waitfor delay \'0:0:__TIME__\'--\n',
 '1) or pg_sleep(__TIME__)--\n',
 '||(elt(-3+5,bin(15),ord(10),hex(char(45))))\n',
 '"hi"") or (""a""=""a"\n',
 'delete\n',
 'like\n',
 '" or sleep(__TIME__)#\n',
 'pg_sleep(__TIME__)--\n',
 '*(|(objectclass=*))\n',
 'declare @q nvarchar (200) 0x730065006c00650063 ...\n']

In [29]:
sql_lines_Generic=clean_sqli_data(sql_lines_Generic)

In [30]:
sql_lines_Generic[:15] # after cleaning

[" )  or  ( 'x' = 'x",
 ' or 1 = 1',
 "; execute immediate 'sel' || 'ect us' || 'er'",
 'benchmark ( 10000000,md5 ( 1  )  )  #',
 'update',
 '";waitfor delay \'0:0:__time__\'--',
 '1 )  or pg_sleep ( __time__ ) --',
 '|| ( elt ( -3+5,bin ( 15 ) ,ord ( 10 ) ,hex ( char ( 45  )  )    )  )  ',
 '"hi"" )  or  ( ""a"" = ""a"',
 'delete',
 'like',
 '" or sleep ( __time__ ) #',
 'pg_sleep ( __time__ ) --',
 '* ( | ( objectclass = *  )  )  ',
 'declare @q nvarchar  ( 200 )  0x730065006c00650063 ...']

### File_name: scottparker_ml_sqli_src__trainingdata_plain.txt
##### Comments: 
removed last 22 records containing URLs
#### Source: https://github.com/tungpv98/Detect-Sql-Injection-by-Machine-Learning/tree/ebeb3287931677a3a42f11a7a08dbc13e374ff05/Sql-Injection/source/trainingdata

In [31]:

# function to remove stopwords

stop_words = set(stopwords.words('english')) 

def fun_remove_stop_words(posts):

    filtered=''
    
    for x in posts.split(' '):
        if x not in stop_words:
            filtered+=' '+x
    
    return filtered

In [32]:
path='./data/'
file="plain.txt"

In [33]:
#read benign data

df = pd.read_csv(os.path.join(path,file), sep='Aw3s0meSc0t7', names=['benign'], header=None, engine='python')

In [34]:
df.head()

,benign
0,Add plain text here
1,“Ne te quaesiveris extra.”
2,“Man is his own star; and the soul that can
3,"Render an honest and a perfect man,"
4,"Commands all light, all influence, all fate;"


In [35]:
plain_text=df['benign'].values  # get sentences

In [36]:
plain_text[:5]

array(['Add plain text here', '“Ne te quaesiveris extra.”',
       '“Man is his own star; and the soul that can',
       'Render an honest and a perfect man,',
       'Commands all light, all influence, all fate;'], dtype=object)

In [37]:
#plain_text=plain_text[:-22] # removed last 22 records that were urls [Already Removed]

In [38]:
len(plain_text)

6936

In [39]:
# convert from list to string

data=''
for x in plain_text:
    data+=" " + x

In [40]:
type(data)

str

In [41]:
data=fun_remove_stop_words(data)  # remove stop words
data=data.split('.')              # split sentences

In [42]:
# seperate words inside tags

for i in range(len(data)):
    data[i]=data[i].replace('<', ' <')
    data[i]=data[i].replace('>', '> ')
    data[i]=data[i].replace('=', ' = ')

In [43]:
data[:5]

['  Add plain text “Ne te quaesiveris extra',
 '” “Man star; soul Render honest perfect man, Commands light, influence, fate; Nothing falls early late',
 ' Our acts angels are, good ill, Our fatal shadows walk us still',
 '” Epilogue Beaumont Fletcher’s Honest Man’s Fortune Cast bantling rocks, Suckle she-wolf’s teat; Wintered hawk fox, Power speed hands feet',
 ' I read day verses written eminent painter original conventional']

Statistics

In [44]:
print("Benign records: %2i" %len(data))

Benign records: 5369


In [45]:
# read self created benign data

path='./benign_for_training.txt'
benign_data=[]
f = open(path, "r")
for x in f:
    benign_data.append(x)


In [46]:
# read self created sqli data

path='./sqli_for_training.txt'
sqli_data=[]
f = open(path, "r")
for x in f:
    sqli_data.append(x)


In [47]:
len(benign_data)

440

In [48]:
benign_sentence=[]
for i in benign_data:
    sentences=i.split('.')
    
    for sentence in sentences:
        benign_sentence.append(sentence)

In [49]:
len(benign_sentence)

1115

In [50]:
len(sqli_data)

290

In [51]:
print(f"SQL fuzzing : {len(sql_lines_fuzzing)}  camoufl4gs : {len(sql_lines_camoufl4g3)} parsed : {len(sql_lines_bypasses)} owasp : {len(sql_lines_owasp)} generic : {len(sql_lines_Generic)} \n total sql injection data : {len(sql_lines_Generic)+len(sql_lines_owasp)+len(sql_lines_bypasses)+len(sql_lines_camoufl4g3)+len(sql_lines_fuzzing)} ")

SQL fuzzing : 86  camoufl4gs : 77 parsed : 474 owasp : 6692 generic : 308 
 total sql injection data : 7637 


#### combination of all sqli attacks

In [52]:
all_sqli_sentence=sql_lines_owasp+sql_lines_bypasses+sql_lines_camoufl4g3+sql_lines_fuzzing+sql_lines_Generic

In [53]:
len(all_sqli_sentence)

7637

In [54]:
# replace numeric values by a keyword 'numeric'
def optional_numeric_to_numeric(all_sqli_sentence):
    
    for i in range(len(all_sqli_sentence)):
        
        all_sqli_sentence[i]=all_sqli_sentence[i].replace('1 ', 'numeric')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(' 1', 'numeric')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace("'1 ", "'numeric ")
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 1'", " numeric'")
        all_sqli_sentence[i]=all_sqli_sentence[i].replace('1,', 'numeric,')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace("1\ ", 'numeric')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace("‘1", '‘numeric')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 2 ", " numeric ")
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(' 3 ', ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(' 3--', ' numeric--')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 4 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 5 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(' 6 ', ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 7 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 8 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace('1234', ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace("22", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 8 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 200 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace("23 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace('"1', '"numeric')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace('1"', '"numeric')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace("7659", 'numeric')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 37 ", ' numeric ')
        all_sqli_sentence[i]=all_sqli_sentence[i].replace(" 45 ", ' numeric ')
    
    return all_sqli_sentence


In [55]:
all_sqli_sentence

['\' or "',
 '-- or # ',
 "' or '1",
 "' or 1 -- -",
 '" or ""  =  "',
 '" or 1  =  1 -- -',
 "' or ''  =  '",
 "' = '",
 "'like'",
 "' = 0--+",
 ' or 1 = 1',
 "' or 'x' = 'x",
 "' and id is null; --",
 "'''''''''''''union select '2",
 'and 1',
 'and 0',
 'and true',
 'and false',
 '1-false',
 '1-true',
 '1*56',
 '-2',
 "1' order by 1--+",
 "1' order by 2--+",
 "1' order by 3--+",
 "1' order by 1,2--+",
 "1' order by 1,2,3--+",
 "1' group by 1,2,--+",
 "1' group by 1,2,3--+",
 "' group by columnnames having 1 = 1 --",
 "-1' union select 1,2,3--+",
 "' union select sum ( columnname  )  from tablename --",
 '-1 union select 1 into @,@',
 '-1 union select 1 into @,@,@',
 '1 and  ( select * from users )   =  1\t',
 "' and mid ( version (  ) ,1,1 )   =  '5';",
 "' and 1 in  ( select min ( name )  from sysobjects where xtype  =  'u' and name > '.' )  --",
 ', ( select * from  ( select ( sleep ( 10  )  )   ) a ) ',
 '%2c ( select * from  ( select ( sleep ( 10  )  )   ) a ) ',
 "';waitfor dela

#### Combination of benign an attack data

In [56]:
import pandas as pd

In [57]:
# give labels to sql data

values=[]
for i in all_sqli_sentence:
    values.append((i,1))

In [58]:
# give labels to benign data

for i in data:
    values.append((i,0))

In [59]:
len(all_sqli_sentence)+len(data)

13006

In [60]:
len(values)

13006

In [61]:
#Add Own data to values
for i in benign_sentence:
    values.append((i,0))

In [62]:
len(values)

14121

In [63]:
#Adding Own data to values
for i in sqli_data:
    values.append((i,1))

In [64]:
len(values)

14411

In [65]:
values[1]

('-- or # ', 1)

In [66]:
# convert to dataframe

df=pd.DataFrame(values,columns=['Sentence','Label'])



In [67]:
df.head()

,Sentence,Label
0,"' or """,1
1,-- or #,1
2,' or '1,1
3,' or 1 -- -,1
4,""" or """" = """,1


### Save data as csv

In [68]:
df.to_csv('sqli.csv', index=False, encoding='utf-16')

In [69]:
df=pd.read_csv('sqli.csv',encoding='utf-16')
df.drop_duplicates(inplace= True, ignore_index=True)

In [70]:
# vectorization of data

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer( min_df=2, max_df=0.7, stop_words=stopwords.words('english'))
posts = vectorizer.fit_transform(df['Sentence'].values.astype('U')).toarray()



In [71]:
posts

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [72]:
posts.shape

(13441, 6835)

In [73]:
transformed_posts=pd.DataFrame(posts)

In [74]:
df=pd.concat([df,transformed_posts],axis=1)

In [75]:
df.head()

,Sentence,Label,0,1,2,3,4,5,6,7,...,6825,6826,6827,6828,6829,6830,6831,6832,6833,6834
0,"' or """,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,-- or #,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,' or '1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,' or 1 -- -,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,""" or """" = """,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [76]:
X=df[df.columns[2:]]

In [77]:
X.shape

(13441, 6835)

In [78]:
y=df['Label']

In [79]:
from sklearn.model_selection import train_test_split

In [80]:
# split train test data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [81]:
print('X_train.shape',X_train.shape,'#y_train.shape',y_train.shape,"#X_test.shape",X_test.shape,"#y_test.shape",y_test.shape)

X_train.shape (10752, 6835) #y_train.shape (10752,) #X_test.shape (2689, 6835) #y_test.shape (2689,)


In [82]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0).fit(X_train, y_train)

In [83]:
from sklearn.metrics import accuracy_score

In [84]:
y_pred=clf.predict(X_test)

In [85]:
accuracy_score(y_test, y_pred)

0.9721085905541094

In [86]:
for i,j in zip(y_test,y_pred):
    print(i==j)

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True

In [87]:
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier

In [88]:
input_dim = X_train.shape[1]  # Number of features

model = Sequential()

model.add(tensorflow.keras.layers.Dense(1024, input_dim=input_dim, activation='relu'))
model.add(tensorflow.keras.layers.Dense(512, activation='relu'))

model.add(tensorflow.keras.layers.Dense(256, activation='relu'))

model.add(tensorflow.keras.layers.Dense(128, activation='relu'))

# model.add(keras.layers.Dense(20,  activation='relu'))
# model.add(keras.layers.Dense(10,  activation='tanh'))
# # model.add(layers.Flatten())
# model.add(keras.layers.Dense(1024, activation='relu'))

model.add(tensorflow.keras.layers.BatchNormalization())
model.add(tensorflow.keras.layers.Dropout(0.5))
model.add(tensorflow.keras.layers.Dense(1, activation='sigmoid'))





In [89]:
model.compile(loss='binary_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense (Dense)                (None, 1024)              7000064   
_________________________________________________________________
dense_1 (Dense)              (None, 512)               524800    
_________________________________________________________________
dense_2 (Dense)              (None, 256)               131328    
_________________________________________________________________
dense_3 (Dense)              (None, 128)               32896     
_________________________________________________________________
batch_normalization (BatchNo (None, 128)               512       
_________________________________________________________________
dropout (Dropout)            (None, 128)               0         
_________________________________________________________________
dense_4 (Dense)              (None, 1)                 1

In [90]:
#import tensorflow as tf

#class myCallback(tf.keras.callbacks.Callback):
#     def on_epoch_end(self, epoch, logs={}):
#         if(logs.get('acc')>0.98):
#             print( "Reached 99.8% accuracy so cancelling training!")
#             self.model.stop_training=True
#stoptrcallback = myCallback()

In [91]:
print()
classifier_nn = model.fit(X_train,y_train,
                    epochs=10,
                    verbose=True,
                    validation_data=(X_test, y_test),
                    batch_size=1024,)



Epoch 1/10
11/11 [==============================] - 6s 545ms/step - loss: 0.3792 - accuracy: 0.8204 - val_loss: 0.4439 - val_accuracy: 0.9580
Epoch 2/10
11/11 [==============================] - 6s 579ms/step - loss: 0.0724 - accuracy: 0.9769 - val_loss: 0.2736 - val_accuracy: 0.9598
Epoch 3/10
11/11 [==============================] - 6s 584ms/step - loss: 0.0405 - accuracy: 0.9832 - val_loss: 0.2088 - val_accuracy: 0.9624
Epoch 4/10
11/11 [==============================] - 7s 607ms/step - loss: 0.0338 - accuracy: 0.9823 - val_loss: 0.1826 - val_accuracy: 0.9650
Epoch 5/10
11/11 [==============================] - 8s 711ms/step - loss: 0.0304 - accuracy: 0.9863 - val_loss: 0.1611 - val_accuracy: 0.9617
Epoch 6/10
11/11 [==============================] - 7s 621ms/step - loss: 0.0309 - accuracy: 0.9851 - val_loss: 0.1462 - val_accuracy: 0.9792
Epoch 7/10
11/11 [==============================] - 7s 680ms/step - loss: 0.0295 - accuracy: 0.9836 - val_loss: 0.1388 - val_accuracy: 0.9784
Epoch

In [92]:
pred=model.predict(X_test)

In [93]:
for i in range(len(pred)):
    if pred[i]>0.5:
        pred[i]=1
    elif pred[i]<=0.5:
        pred[i]=0

In [94]:
accuracy_score(y_test,pred)

0.9616957976943101

In [95]:
for i,j in zip(y_test,pred):
    print(i==j)

[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]


[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[False]
[ True]
[ True]
[False]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
[ True]


#### Saving Model

In [96]:
from tensorflow.keras.models import load_model
import pickle
print("###Saving Model###")
model.save('Final-SQLI-Model.h5')
print('###Saving Vectorizer###')
with open('Final-SQLI-Vectorizer', 'wb') as fin:
    pickle.dump(vectorizer, fin)


###Saving Model###
###Saving Vectorizer###


### Confusion Matrix
#### Assumption:
1. SQLI attack type is positive

In [97]:
def accuracy_function(tp,tn,fp,fn):
    
    accuracy = (tp+tn) / (tp+tn+fp+fn)
    
    return accuracy

In [98]:
def precision_function(tp,fp):
    
    precision = tp / (tp+fp)
    
    return precision

In [99]:
def recall_function(tp,fn):
    
    recall=tp / (tp+fn)
    
    return recall

In [100]:
def confusion_matrix(truth,predicted):
    
    true_positive = 0
    true_negative = 0
    false_positive = 0
    false_negative = 0
    
    for true,pred in zip(truth,predicted):
        
        if true == 1:
            if pred == true:
                true_positive += 1
            elif pred != true:
                false_negative += 1

        elif true == 0:
            if pred == true:
                true_negative += 1
            elif pred != true:
                false_positive += 1
            
    accuracy=accuracy_function(true_positive, true_negative, false_positive, false_negative)
    precision=precision_function(true_positive, false_positive)
    recall=recall_function(true_positive, false_negative)
    
    return (accuracy,
            precision,
           recall)

In [101]:
accuracy,precision,recall=confusion_matrix(y_test,pred)

In [102]:
print(" Accuracy : {0} \n Precision : {1} \n Recall : {2}".format(accuracy, precision, recall))

 Accuracy : 0.9616957976943101 
 Precision : 0.9911684782608695 
 Recall : 0.9418979987088444


In [103]:

from sklearn.metrics import precision_score
precision_score(y_test, pred)

0.9911684782608695

In [104]:
from sklearn.metrics import recall_score
recall_score(y_test, pred)


0.9418979987088444

### Classifying user data

     This cell is also in file user_Data_predict.py No need to run this cell

In [111]:

import tensorflow.keras as keras
from tensorflow.keras.models import load_model
import pickle

mymodel = load_model('Final-SQLI-Model.h5')
myvectorizer = pickle.load(open("Final-SQLI-Vectorizer", 'rb'))




def clean_data(input_val):

    input_val=input_val.replace('\n', '')
    input_val=input_val.replace('%20', ' ')
    input_val=input_val.replace('=', ' = ')
    input_val=input_val.replace('((', ' (( ')
    input_val=input_val.replace('))', ' )) ')
    input_val=input_val.replace('(', ' ( ')
    input_val=input_val.replace(')', ' ) ')
    input_val=input_val.replace('1 ', 'numeric')
    input_val=input_val.replace(' 1', 'numeric')
    input_val=input_val.replace("'1 ", "'numeric ")
    input_val=input_val.replace(" 1'", " numeric'")
    input_val=input_val.replace('1,', 'numeric,')
    input_val=input_val.replace(" 2 ", " numeric ")
    input_val=input_val.replace(' 3 ', ' numeric ')
    input_val=input_val.replace(' 3--', ' numeric--')
    input_val=input_val.replace(" 4 ", ' numeric ')
    input_val=input_val.replace(" 5 ", ' numeric ')
    input_val=input_val.replace(' 6 ', ' numeric ')
    input_val=input_val.replace(" 7 ", ' numeric ')
    input_val=input_val.replace(" 8 ", ' numeric ')
    input_val=input_val.replace('1234', ' numeric ')
    input_val=input_val.replace("22", ' numeric ')
    input_val=input_val.replace(" 8 ", ' numeric ')
    input_val=input_val.replace(" 200 ", ' numeric ')
    input_val=input_val.replace("23 ", ' numeric ')
    input_val=input_val.replace('"1', '"numeric')
    input_val=input_val.replace('1"', '"numeric')
    input_val=input_val.replace("7659", 'numeric')
    input_val=input_val.replace(" 37 ", ' numeric ')
    input_val=input_val.replace(" 45 ", ' numeric ')

    return input_val








def predict_sqli_attack(input_val=0):
    
    repeat=True
    
    beautify=''
    for i in range(20):
        beautify+= "="

    if inputval==0:
        print(beautify) 
        input_val=input("Give me some data to work on : ")
        print(beautify)

    
    if input_val== '0':
        repeat=False
    
    
    input_val=clean_data(input_val)
    input_val=[input_val]



    input_val=myvectorizer.transform(input_val).toarray()

    result=mymodel.predict(input_val)

    print(beautify)
    
    
    if repeat == True:
        
        if result>0.5:
            print(result,"ALERT :::: This can be SQL injection")


        elif result<=0.5:
            print(result,"It seems to be safe")
            
        print(beautify)
            
        predict_sqli_attack()
            
    elif repeat == False:
        print( " Good Bye ")

 



In [112]:
# check for sql injection cheat sheet





predict_sqli_attack()

Give me some data to work on : 0
 Good Bye 


In [108]:
# check for normal data





predict_sqli_attack()

Give me some data to work on : a=bfsdkksgksgr&vhsdh=sbgjhdr
[[0.46545765]] It seems to be safe
Give me some data to work on : fhdbdsjh=2312313&jhbcv=ngsbks
[[0.46545765]] It seems to be safe
Give me some data to work on : 0
 Good Bye 
